# GRU Stock Price Prediction Pipeline
**Multi-window hyperparameter search | Nifty50 | Quantile Loss**

- **Target**: `target_pct_change` (daily return fraction, e.g. 0.02 = +2%)
- **Loss**: Quantile (pinball) loss — 7 quantiles
- **Windows**: 7, 10, 15, 30 days × 3 model sizes (12 runs total)
- **Metrics**:
  - `MAPE_price` — MAPE on reconstructed price (base × (1+pred) vs actual)
  - `MSE/RMSE/MAE_pct` — error on pct-change ×100 (percentage-point errors)
  - `F1_macro/Acc/Prec_macro/Recall_macro/DA` — direction classification (macro avg)
- **Split**: train ≤ 2024-12-31 | val 2025-01–06 | test ≥ 2025-07-01

### Bug Fixes vs v1
- **recall=1 bug**: now uses `average='macro'` so both UP and DOWN classes are weighted equally
- **Missing predicted line in test plots**: x-axis uses sequential integer index, not broken date conversion
- **Wrong ΔPrice metric**: MSE/RMSE/MAE now computed on `pct_change × 100` (percentage-point space), not `adj_close × pct_change` (adj_close is z-scored so that product is meaningless)

In [1]:
%%capture
!pip install scikit-learn matplotlib seaborn tqdm

In [2]:
import os, sys, json, math, random, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score
)

try:
    from torch.cuda.amp import GradScaler, autocast
    AMP_AVAILABLE = True
except ImportError:
    AMP_AVAILABLE = False

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB
PyTorch: 2.10.0+cu128


In [3]:
# ── Paths ──────────────────────────────────────────────────────────────
DATA_PATH = Path('dataset.csv')
if not DATA_PATH.exists():
    # Fallback if running on Kaggle
    DATA_PATH = Path('/kaggle/input/dataset/dataset.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError('dataset.csv not found!')
print(f'Dataset: {DATA_PATH}')

TICKER_PATH = Path('nifty50_ticker.csv')
if not TICKER_PATH.exists():
    TICKER_PATH = Path('/kaggle/input/dataset/nifty50_ticker.csv')
if not TICKER_PATH.exists():
    TICKER_PATH = Path('/kaggle/input/nifty50-dataset/nifty50_ticker.csv')
assert TICKER_PATH.exists(), f'nifty50_ticker.csv not found!'
print(f'Ticker : {TICKER_PATH}')

OUTPUT_DIR = Path('/kaggle/working/lstm_outputs')
for _sub in ['checkpoints', 'plots', 'results']:
    (OUTPUT_DIR / _sub).mkdir(parents=True, exist_ok=True)

# ── Date splits (identical to TFT pipeline) ────────────────────────────
TRAIN_END  = pd.Timestamp('2024-12-31')
VAL_START  = pd.Timestamp('2025-01-01')
VAL_END    = pd.Timestamp('2025-06-30')
TEST_START = pd.Timestamp('2025-07-01')

# ── Quantiles (7, identical to TFT output_size=7) ─────────────────────
QUANTILES  = [0.10, 0.25, 0.40, 0.50, 0.60, 0.75, 0.90]
MEDIAN_IDX = QUANTILES.index(0.50)   # = 3

# ── Hyperparameter search space ───────────────────────────────────────
WINDOWS: List[int] = [7, 10, 15, 30]

MODEL_CONFIGS: Dict[str, Dict] = {
    'small':  dict(hidden_size=32,  num_layers=1, dropout=0.10, lr=1e-3,  batch_size=256),
    'medium': dict(hidden_size=64,  num_layers=2, dropout=0.20, lr=1e-3,  batch_size=512),
    'large':  dict(hidden_size=128, num_layers=2, dropout=0.30, lr=5e-4,  batch_size=512),
}

# ── Training ──────────────────────────────────────────────────────────
MAX_EPOCHS  = 50
PATIENCE    = 10
NUM_WORKERS = 2
GRAD_CLIP   = 1.0
USE_AMP     = AMP_AVAILABLE and DEVICE.type == 'cuda'

# ── Column names ──────────────────────────────────────────────────────
TARGET_COL = 'target_pct_change'
PRICE_COL  = 'Adj Close'
DATE_COL   = 'Date'
SYMBOL_COL = 'Symbol'

EXCLUDE_COLS = {
    'Date', 'Symbol', 'Open', 'High', 'Low', 'Close',
    'Adj Close', 'Volume', 'symbol_base', 'time_idx',
}

EPS = 1e-8

# ── Plot aesthetics ────────────────────────────────────────────────────
PALETTE = sns.color_palette('tab10', 10)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except Exception:
    plt.style.use('seaborn-darkgrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 9, 'axes.labelsize': 8,
    'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 7,
})

print(f'Config loaded. Windows={WINDOWS}, MaxEpochs={MAX_EPOCHS}, AMP={USE_AMP}')


Dataset: /kaggle/input/datasets/sriramparuchuri/new-data/tft_ready (2).csv
Config loaded. Windows=[7, 10, 15, 30], MaxEpochs=50, AMP=True


In [4]:
# ─── Data Loading & Feature Engineering ──────────────────────────────────
# Feature engineering identical to tft.py (load_and_prepare_dataframe)

def load_and_prepare(path: Path) -> Tuple[pd.DataFrame, List[str]]:
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = df.dropna(subset=[DATE_COL])
    df = df.sort_values([SYMBOL_COL, DATE_COL]).reset_index(drop=True)
    df = df.drop_duplicates([SYMBOL_COL, DATE_COL])

    # Calendar covariates — same as tft.py
    df['dow']            = df[DATE_COL].dt.weekday.astype(np.float32)
    df['dom']            = df[DATE_COL].dt.day.astype(np.float32)
    df['month']          = df[DATE_COL].dt.month.astype(np.float32)
    df['is_month_start'] = df[DATE_COL].dt.is_month_start.astype(np.float32)
    df['is_month_end']   = df[DATE_COL].dt.is_month_end.astype(np.float32)

    num_cols  = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    feat_cols = [c for c in num_cols if c not in EXCLUDE_COLS]

    # Target first — autoregressive input (mirrors TFT unknown_reals[0])
    if TARGET_COL in feat_cols:
        feat_cols = [TARGET_COL] + [c for c in feat_cols if c != TARGET_COL]

    df[feat_cols] = df[feat_cols].fillna(0.0)

    print(f'Loaded : {df.shape[0]:,} rows | {df[SYMBOL_COL].nunique()} symbols')
    print(f'Dates  : {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}')
    print(f'Features ({len(feat_cols)}): {feat_cols}')
    return df, feat_cols


df_raw, FEATURE_COLS = load_and_prepare(DATA_PATH)
N_FEATURES = len(FEATURE_COLS)
SYMBOLS    = sorted(df_raw[SYMBOL_COL].unique())
print(f'\nSymbols ({len(SYMBOLS)}): {[s.replace(".NS","") for s in SYMBOLS]}')

Loaded : 71,938 rows | 50 symbols
Dates  : 2020-03-12 → 2026-03-27
Features (34): ['target_pct_change', 'adj_ret_1d', 'price_range_pct', 'gap_pct', 'rolling_volatility_5d', 'adjclose_sma_ratio_5', 'volume_momentum_5', 'adjclose_sma_ratio_10', 'volume_momentum_10', 'adjclose_sma_ratio_20', 'volume_momentum_20', 'adjclose_sma_ratio_50', 'volume_momentum_50', 'rsi_14', 'macd', 'macd_signal', 'macd_diff', 'direct_news_pos', 'direct_news_neu', 'direct_news_neg', 'sectoral_news_pos', 'sectoral_news_neu', 'sectoral_news_neg', 'global_news_pos', 'global_news_neu', 'global_news_neg', 'direct_news_count', 'sectoral_news_count', 'global_news_count', 'dow', 'dom', 'month', 'is_month_start', 'is_month_end']

Symbols (50): ['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK', 'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BEL', 'BHARTIARTL', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT', 'ETERNAL', 'GRASIM', 'HCLTECH', 'HDFCBANK', 'HDFCLIFE', 'HINDALCO', 'HINDUNILVR', 'ICICIBANK', 'INDIGO', 

In [5]:
# ─── Dataset Class ────────────────────────────────────────────────────────

class StockWindowDataset(Dataset):
    """
    Sliding-window dataset for multi-symbol stock prediction.

    Each sample at time t for symbol s:
      x       : features[t-window : t]   → (window, n_features)
      y       : target_pct_change[t]     → scalar (raw fraction, e.g. 0.02)
      base_px : Adj_Close[t-1]           → standardised z-score (for MAPE only)
      true_px : Adj_Close[t]             → standardised z-score (for MAPE only)

    NOTE: Adj Close in the dataset is z-score standardised per symbol.
          Do NOT multiply pct_change by base_px for ΔPrice — use return space.

    Split by prediction date:
      train : date ≤ TRAIN_END
      val   : VAL_START ≤ date ≤ VAL_END
      test  : date ≥ TEST_START
    """

    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: List[str],
        window_size: int,
        split: str,
        scaler: Optional[StandardScaler] = None,
        fit_scaler: bool = False,
    ):
        assert split in ('train', 'val', 'test')
        self.window_size  = window_size
        self.feature_cols = feature_cols

        xs, ys, bpxs, tpxs = [], [], [], []
        syms_list, date_list = [], []
        train_feats_list: List[np.ndarray] = []

        for sym, grp in df.groupby(SYMBOL_COL, sort=True):
            grp = grp.sort_values(DATE_COL).reset_index(drop=True)
            if len(grp) < window_size + 1:
                continue

            feats   = grp[feature_cols].values.astype(np.float32)
            targets = grp[TARGET_COL].values.astype(np.float32)
            prices  = grp[PRICE_COL].values.astype(np.float32)
            dt_arr  = grp[DATE_COL].values

            feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

            if fit_scaler:
                tmask = pd.DatetimeIndex(dt_arr) <= TRAIN_END
                if tmask.any():
                    train_feats_list.append(feats[tmask])

            for t in range(window_size, len(grp)):
                d = pd.Timestamp(dt_arr[t])
                in_split = {
                    'train': d <= TRAIN_END,
                    'val':   VAL_START <= d <= VAL_END,
                    'test':  d >= TEST_START,
                }[split]
                if not in_split:
                    continue
                xs.append(feats[t - window_size : t])
                ys.append(targets[t])
                bpxs.append(prices[t - 1])
                tpxs.append(prices[t])
                syms_list.append(sym)
                date_list.append(d)

        # Fit scaler on train features only
        if fit_scaler:
            if train_feats_list:
                scaler = StandardScaler()
                scaler.fit(np.vstack(train_feats_list))
        self.scaler = scaler

        if len(xs) > 0:
            xs_arr = np.stack(xs).astype(np.float32)      # (N, W, F)
            if self.scaler is not None:
                N, W, F = xs_arr.shape
                xs_arr  = self.scaler.transform(
                    xs_arr.reshape(-1, F)
                ).reshape(N, W, F)
            self.xs = xs_arr
        else:
            self.xs = np.zeros((0, window_size, len(feature_cols)), np.float32)

        self.ys       = np.array(ys,        np.float32)
        self.base_pxs = np.array(bpxs,      np.float32)
        self.true_pxs = np.array(tpxs,      np.float32)
        self.symbols  = np.array(syms_list,  object)
        self.dates    = np.array(date_list,  object)

        n_sym = len(np.unique(self.symbols)) if len(self.symbols) > 0 else 0
        print(f'  [{split:5s}] {len(self.xs):7,} samples | {n_sym:3d} symbols | window={window_size}')

    def __len__(self):
        return len(self.xs)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.xs[idx]),
            torch.tensor(self.ys[idx],       dtype=torch.float32),
            torch.tensor(self.base_pxs[idx], dtype=torch.float32),
            torch.tensor(self.true_pxs[idx], dtype=torch.float32),
        )


def build_datasets(
    df: pd.DataFrame,
    feature_cols: List[str],
    window_size: int,
) -> Tuple['StockWindowDataset', 'StockWindowDataset', 'StockWindowDataset', StandardScaler]:
    print(f'\nBuilding datasets (window={window_size}) …')
    train_ds = StockWindowDataset(df, feature_cols, window_size, 'train', fit_scaler=True)
    val_ds   = StockWindowDataset(df, feature_cols, window_size, 'val',   scaler=train_ds.scaler)
    test_ds  = StockWindowDataset(df, feature_cols, window_size, 'test',  scaler=train_ds.scaler)
    return train_ds, val_ds, test_ds, train_ds.scaler

In [6]:
# ─── GRU Model ────────────────────────────────────────────────────────────

class GRUQuantile(nn.Module):
    """
    Deep GRU with multi-quantile output head.

    Architecture:
        GRU Layer 1 (input → hidden)          + Tanh + Dropout
        GRU Layer 2 (hidden → hidden)         + Tanh + Dropout  + Residual
        GRU Layer 3 (hidden → hidden)         + Tanh + Dropout  + Residual
        LayerNorm(hidden)
        Linear(hidden → hidden // 2)  + GELU
        Linear(hidden // 2 → num_quantiles)

    - 3 stacked GRU layers with explicit Tanh non-linearity between them
    - Residual (skip) connections on layers 2 & 3 for stable gradient flow
    - Two-layer projection head with GELU for non-linear quantile mapping
    """

    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 dropout=0.2, num_quantiles=7):
        super().__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.num_quantiles = num_quantiles

        # ── 3 stacked GRU layers with inter-layer non-linearity ──
        self.gru1 = nn.GRU(input_size,  hidden_size, num_layers=1, batch_first=True)
        self.act1  = nn.Tanh()
        self.drop1 = nn.Dropout(dropout)

        self.gru2 = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act2  = nn.Tanh()
        self.drop2 = nn.Dropout(dropout)

        self.gru3 = nn.GRU(hidden_size, hidden_size, num_layers=1, batch_first=True)
        self.act3  = nn.Tanh()
        self.drop3 = nn.Dropout(dropout)

        # ── Projection head ──
        self.norm  = nn.LayerNorm(hidden_size)
        self.fc1   = nn.Linear(hidden_size, hidden_size // 2)
        self.gelu  = nn.GELU()
        self.drop4 = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden_size // 2, num_quantiles)

        self._init_weights()

    def _init_weights(self):
        for gru_layer in [self.gru1, self.gru2, self.gru3]:
            for name, p in gru_layer.named_parameters():
                if 'weight_ih' in name:   nn.init.xavier_uniform_(p)
                elif 'weight_hh' in name: nn.init.orthogonal_(p)
                elif 'bias' in name:      nn.init.zeros_(p)
        for fc in [self.fc1, self.head]:
            nn.init.xavier_uniform_(fc.weight)
            nn.init.zeros_(fc.bias)

    def forward(self, x):
        # Layer 1
        out, _ = self.gru1(x)
        out    = self.drop1(self.act1(out))

        # Layer 2 + residual
        res    = out
        out, _ = self.gru2(out)
        out    = self.drop2(self.act2(out)) + res

        # Layer 3 + residual
        res    = out
        out, _ = self.gru3(out)
        out    = self.drop3(self.act3(out)) + res

        # Projection head (last timestep)
        last   = self.norm(out[:, -1, :])
        h      = self.drop4(self.gelu(self.fc1(last)))
        return  self.head(h)                  # (B, Q)

    def print_summary(self, window: int, cfg: str):
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"\n{'='*56}")
        print(f" GRU Quantile  |  window={window}  |  config={cfg}")
        print(f"{'='*56}")
        print(self)
        print(f"\n Total params    : {total:,}")
        print(f" Trainable params: {train:,}")
        print(f"{'='*56}")


In [7]:
# ─── Loss Function & Metrics ──────────────────────────────────────────────

def quantile_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """Pinball (quantile) loss averaged over quantiles and batch."""
    q   = torch.tensor(QUANTILES, dtype=y_pred.dtype, device=y_pred.device)
    err = y_true.unsqueeze(-1) - y_pred    # (B, Q)
    return torch.max(q * err, (q - 1) * err).mean()


def compute_metrics(
    y_true:  np.ndarray,    # actual target_pct_change  (e.g. 0.02 = +2%)
    y_pred:  np.ndarray,    # predicted pct_change (median quantile)
    base_px: np.ndarray,    # Adj_Close[t-1]  — z-scored; used ONLY for MAPE
    true_px: np.ndarray,    # Adj_Close[t]    — z-scored; used ONLY for MAPE
) -> Dict[str, float]:
    """
    Compute all required metrics.

    ── Regression ───────────────────────────────────────────────────────────
    MAPE_price    : MAPE on reconstructed price  (base_px*(1+pred) vs true_px)
                    Works correctly because base_px and true_px preserve the
                    relative scale even when z-scored.

    MSE/RMSE/MAE_pct : error in pct-change space multiplied by 100.
                    e.g. MAE_pct=1.5 means the model is off by 1.5 pct-points
                    on average — directly interpretable without needing actual
                    rupee prices.  (Using base_px*pct_change is WRONG because
                    base_px is a z-score that can be negative or ~0.)

    ── Classification (directional accuracy) ────────────────────────────────
    Rules (per user spec):
      pred > 0 AND actual > 0  → correct  (label 1)
      pred < 0 AND actual < 0  → correct  (label 1)
      opposite signs           → incorrect (label 0)

    Metrics use average='macro' so UP and DOWN classes are weighted equally.
    This prevents the recall=1 degeneracy that occurs when the model predicts
    nearly all UP (because tiny positive predictions outnumber tiny negatives).
    """
    valid = (
        np.isfinite(y_true) & np.isfinite(y_pred)
        & np.isfinite(base_px) & np.isfinite(true_px)
    )
    yt, yp, bp, tp = y_true[valid], y_pred[valid], base_px[valid], true_px[valid]
    if len(yt) == 0:
        return {'n_samples': 0}

    # ── MAPE on reconstructed price ──────────────────────────────────────
    pred_px  = bp * (1.0 + yp)
    denom_px = np.where(np.abs(tp) < EPS, EPS, np.abs(tp))
    mape_px  = float(np.mean(np.abs(pred_px - tp) / denom_px) * 100.0)

    # ── Pct-change error metrics (×100 → percentage-point space) ─────────
    # BUG FIX: do NOT use base_px * pct_change because base_px is z-scored.
    err_pct  = (yp - yt) * 100.0     # percentage-point error
    mse_p    = float(np.mean(err_pct ** 2))
    rmse_p   = float(np.sqrt(mse_p))
    mae_p    = float(np.mean(np.abs(err_pct)))

    # ── Direction classification (macro-averaged) ─────────────────────────
    # BUG FIX: use average='macro' to avoid recall=1 when model predicts
    # almost all UP (tiny positive values dominate near-zero predictions).
    true_up = (yt > 0).astype(int)   # 1=UP, 0=DOWN
    pred_up = (yp > 0).astype(int)

    # Directional accuracy: fraction where predicted direction == actual
    dir_acc = float(np.mean(true_up == pred_up))

    return {
        # ── Primary model-selection metric ──
        'MAPE_price':      mape_px,
        # ── Regression (pct-point space) ───
        'MSE_pct':         mse_p,
        'RMSE_pct':        rmse_p,
        'MAE_pct':         mae_p,
        # ── Classification (macro so both UP/DOWN classes counted) ──
        'F1_macro':        float(f1_score(true_up, pred_up, average='macro',    zero_division=0)),
        'Accuracy':        float(accuracy_score(true_up, pred_up)),
        'Precision_macro': float(precision_score(true_up, pred_up, average='macro', zero_division=0)),
        'Recall_macro':    float(recall_score(true_up, pred_up, average='macro', zero_division=0)),
        'Directional_Acc': dir_acc,
        # ── UP-class only (for reference) ──
        'F1_up':           float(f1_score(true_up, pred_up, average='binary',   zero_division=0)),
        'Precision_up':    float(precision_score(true_up, pred_up, average='binary', zero_division=0)),
        'Recall_up':       float(recall_score(true_up, pred_up, average='binary', zero_division=0)),
        'n_samples':       len(yt),
        'pct_up_true':     float(np.mean(true_up)),     # fraction of actual UP days
        'pct_up_pred':     float(np.mean(pred_up)),     # fraction of predicted UP days
    }


# Column subsets for display
REPORT_COLS = [
    'MAPE_price', 'MSE_pct', 'RMSE_pct', 'MAE_pct',
    'F1_macro', 'Accuracy', 'Precision_macro', 'Recall_macro', 'Directional_Acc',
    'pct_up_true', 'pct_up_pred', 'n_samples'
]

In [8]:
# ─── Training & Evaluation Functions ─────────────────────────────────────

def _train_epoch(model, loader, optimizer, amp_scaler, device):
    model.train()
    total, n = 0.0, 0
    for x, y, _, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        if amp_scaler is not None:
            with autocast():
                loss = quantile_loss(model(x), y)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            amp_scaler.step(optimizer)
            amp_scaler.update()
        else:
            loss = quantile_loss(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total += loss.item(); n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate_ds(model, ds, batch_size, device):
    """Evaluate model on dataset (no shuffle). Returns predictions + metadata."""
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda')
    )
    model.eval()
    total, n = 0.0, 0
    preds, trues, bpxs, tpxs = [], [], [], []
    for x, y, bp, tp in loader:
        out  = model(x.to(device))
        loss = quantile_loss(out, y.to(device))
        total += loss.item(); n += 1
        preds.append(out[:, MEDIAN_IDX].cpu().numpy())
        trues.append(y.numpy())
        bpxs.append(bp.numpy())
        tpxs.append(tp.numpy())
    return {
        'loss':    total / max(n, 1),
        'pred':    np.concatenate(preds)  if preds  else np.array([]),
        'true':    np.concatenate(trues)  if trues  else np.array([]),
        'base_px': np.concatenate(bpxs)  if bpxs   else np.array([]),
        'true_px': np.concatenate(tpxs)  if tpxs   else np.array([]),
        'symbols': ds.symbols,
        'dates':   ds.dates,
    }


def train_model(model, train_ds, val_ds, config, save_path, device):
    """Full training loop with early stopping. Returns loss history dict."""
    bs  = config['batch_size']
    dl  = DataLoader(
        train_ds, bs, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=True
    )
    opt   = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=1e-5)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', 0.5, patience=4, min_lr=1e-6)
    amp_sc = GradScaler() if USE_AMP else None

    best_val, pat = float('inf'), 0
    hist = {'train_loss': [], 'val_loss': []}

    pbar = tqdm(range(1, MAX_EPOCHS + 1), desc='Epochs', leave=True)
    for epoch in pbar:
        t_loss = _train_epoch(model, dl, opt, amp_sc, device)
        v_ev   = evaluate_ds(model, val_ds, bs, device)
        v_loss = v_ev['loss']
        sched.step(v_loss)
        hist['train_loss'].append(t_loss)
        hist['val_loss'].append(v_loss)
        pbar.set_postfix(train=f'{t_loss:.4f}', val=f'{v_loss:.4f}', best=f'{best_val:.4f}')

        if v_loss < best_val - 1e-6:
            best_val = v_loss; pat = 0
            torch.save(model.state_dict(), save_path)
        else:
            pat += 1

        if pat >= PATIENCE:
            pbar.close()
            print(f'  Early stop at epoch {epoch}')
            break

    if Path(save_path).exists():
        model.load_state_dict(torch.load(save_path, map_location=device))
    return hist

In [9]:
# ─── Hyperparameter Search ────────────────────────────────────────────────

all_results:   Dict = {}    # (window, cfg) → result dict
all_histories: Dict = {}    # (window, cfg) → history dict


def run_one(window: int, cfg_name: str) -> Optional[Dict]:
    """Train and evaluate one (window, config) combination."""
    config = MODEL_CONFIGS[cfg_name]
    key    = (window, cfg_name)

    print(f"\n{'='*58}")
    print(f"  Window={window}  |  Config={cfg_name.upper()}  "
          f"hidden={config['hidden_size']}  layers={config['num_layers']}")
    print(f"{'='*58}")

    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    train_ds, val_ds, test_ds, scaler = build_datasets(df_raw, FEATURE_COLS, window)
    if len(train_ds) == 0:
        print('  Skipping: no training samples.')
        return None

    model = GRUQuantile(
        input_size    = N_FEATURES,
        hidden_size   = config['hidden_size'],
        num_layers    = config['num_layers'],
        dropout       = config['dropout'],
        num_quantiles = len(QUANTILES),
    ).to(DEVICE)
    model.print_summary(window, cfg_name)

    save_path = OUTPUT_DIR / 'checkpoints' / f'w{window}_{cfg_name}.pt'
    history   = train_model(model, train_ds, val_ds, config, save_path, DEVICE)
    all_histories[key] = history

    bs       = config['batch_size']
    train_ev = evaluate_ds(model, train_ds, bs, DEVICE)
    val_ev   = evaluate_ds(model, val_ds,   bs, DEVICE)
    test_ev  = evaluate_ds(model, test_ds,  bs, DEVICE)

    def _m(ev):
        return compute_metrics(ev['true'], ev['pred'], ev['base_px'], ev['true_px'])

    result = {
        'window': window, 'config': cfg_name,
        'save_path': str(save_path), 'scaler': scaler,
        'train_ev': train_ev, 'val_ev': val_ev, 'test_ev': test_ev,
        'train_metrics': _m(train_ev),
        'val_metrics':   _m(val_ev),
        'test_metrics':  _m(test_ev),
        'history':       history,
    }
    all_results[key] = result

    vm, tm = result['val_metrics'], result['test_metrics']
    print(f"\n  Val  MAPE={vm.get('MAPE_price',0):.4f}%  "
          f"F1_macro={vm.get('F1_macro',0):.4f}  "
          f"Recall_macro={vm.get('Recall_macro',0):.4f}  "
          f"DA={vm.get('Directional_Acc',0):.4f}")
    print(f"  Test MAPE={tm.get('MAPE_price',0):.4f}%  "
          f"F1_macro={tm.get('F1_macro',0):.4f}  "
          f"Recall_macro={tm.get('Recall_macro',0):.4f}  "
          f"DA={tm.get('Directional_Acc',0):.4f}")
    print(f"  pct_up_pred={tm.get('pct_up_pred',0):.3f}  "
          f"pct_up_true={tm.get('pct_up_true',0):.3f}  "
          f"(ratio should be ~1.0 for non-degenerate model)")
    return result


# ── PHASE 1: Initial training — Window = 10 (all 3 configs) ─────────────
print('=' * 58)
print('  PHASE 1  :  Initial Training  |  Window = 10')
print('=' * 58)
for _cfg in MODEL_CONFIGS:
    run_one(10, _cfg)

  PHASE 1  :  Initial Training  |  Window = 10

  Window=10  |  Config=SMALL  hidden=32  layers=1

Building datasets (window=10) …
  [train]  56,296 samples |  49 symbols | window=10
  [val  ]   6,027 samples |  49 symbols | window=10
  [test ]   9,115 samples |  50 symbols | window=10

 GRU Quantile  |  window=10  |  config=small
GRUQuantile(
  (gru): GRU(34, 32, batch_first=True)
  (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.1, inplace=False)
  (head): Linear(in_features=32, out_features=7, bias=True)
)

 Total params    : 6,823
 Trainable params: 6,823


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 49

  Val  MAPE=8.4705%  F1_macro=0.4978  Recall_macro=0.5137  DA=0.5162
  Test MAPE=17.2338%  F1_macro=0.4847  Recall_macro=0.5097  DA=0.5028
  pct_up_pred=0.704  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=10  |  Config=MEDIUM  hidden=64  layers=2

Building datasets (window=10) …
  [train]  56,296 samples |  49 symbols | window=10
  [val  ]   6,027 samples |  49 symbols | window=10
  [test ]   9,115 samples |  50 symbols | window=10

 GRU Quantile  |  window=10  |  config=medium
GRUQuantile(
  (gru): GRU(34, 64, num_layers=2, batch_first=True, dropout=0.2)
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.2, inplace=False)
  (head): Linear(in_features=64, out_features=7, bias=True)
)

 Total params    : 44,743
 Trainable params: 44,743


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 36

  Val  MAPE=8.4511%  F1_macro=0.5029  Recall_macro=0.5062  DA=0.5074
  Test MAPE=17.2085%  F1_macro=0.4854  Recall_macro=0.4901  DA=0.4874
  pct_up_pred=0.580  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=10  |  Config=LARGE  hidden=128  layers=2

Building datasets (window=10) …
  [train]  56,296 samples |  49 symbols | window=10
  [val  ]   6,027 samples |  49 symbols | window=10
  [test ]   9,115 samples |  50 symbols | window=10

 GRU Quantile  |  window=10  |  config=large
GRUQuantile(
  (gru): GRU(34, 128, num_layers=2, batch_first=True, dropout=0.3)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.3, inplace=False)
  (head): Linear(in_features=128, out_features=7, bias=True)
)

 Total params    : 163,207
 Trainable params: 163,207


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 45

  Val  MAPE=8.4484%  F1_macro=0.5162  Recall_macro=0.5181  DA=0.5173
  Test MAPE=17.1935%  F1_macro=0.4965  Recall_macro=0.4973  DA=0.4992
  pct_up_pred=0.444  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)


In [10]:
# ── PHASE 2: Full hyperparameter search — Windows 7, 15, 30 ──────────────
print('=' * 58)
print('  PHASE 2  :  Full Search  |  Windows 7, 15, 30')
print('=' * 58)
for _w in [7, 15, 30]:
    for _cfg in MODEL_CONFIGS:
        run_one(_w, _cfg)

  PHASE 2  :  Full Search  |  Windows 7, 15, 30

  Window=7  |  Config=SMALL  hidden=32  layers=1

Building datasets (window=7) …
  [train]  56,443 samples |  49 symbols | window=7
  [val  ]   6,027 samples |  49 symbols | window=7
  [test ]   9,118 samples |  50 symbols | window=7

 GRU Quantile  |  window=7  |  config=small
GRUQuantile(
  (gru): GRU(34, 32, batch_first=True)
  (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.1, inplace=False)
  (head): Linear(in_features=32, out_features=7, bias=True)
)

 Total params    : 6,823
 Trainable params: 6,823


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 41

  Val  MAPE=8.4813%  F1_macro=0.4883  Recall_macro=0.5067  DA=0.5094
  Test MAPE=17.2552%  F1_macro=0.4801  Recall_macro=0.5070  DA=0.4998
  pct_up_pred=0.711  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=7  |  Config=MEDIUM  hidden=64  layers=2

Building datasets (window=7) …
  [train]  56,443 samples |  49 symbols | window=7
  [val  ]   6,027 samples |  49 symbols | window=7
  [test ]   9,118 samples |  50 symbols | window=7

 GRU Quantile  |  window=7  |  config=medium
GRUQuantile(
  (gru): GRU(34, 64, num_layers=2, batch_first=True, dropout=0.2)
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.2, inplace=False)
  (head): Linear(in_features=64, out_features=7, bias=True)
)

 Total params    : 44,743
 Trainable params: 44,743


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 32

  Val  MAPE=8.4455%  F1_macro=0.5008  Recall_macro=0.5042  DA=0.5054
  Test MAPE=17.2063%  F1_macro=0.4789  Recall_macro=0.4994  DA=0.4932
  pct_up_pred=0.682  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=7  |  Config=LARGE  hidden=128  layers=2

Building datasets (window=7) …
  [train]  56,443 samples |  49 symbols | window=7
  [val  ]   6,027 samples |  49 symbols | window=7
  [test ]   9,118 samples |  50 symbols | window=7

 GRU Quantile  |  window=7  |  config=large
GRUQuantile(
  (gru): GRU(34, 128, num_layers=2, batch_first=True, dropout=0.3)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.3, inplace=False)
  (head): Linear(in_features=128, out_features=7, bias=True)
)

 Total params    : 163,207
 Trainable params: 163,207


...

  Early stop at epoch 33

  Val  MAPE=8.4400%  F1_macro=0.4882  Recall_macro=0.5141  DA=0.5172
  Test MAPE=16.7944%  F1_macro=0.4671  Recall_macro=0.5085  DA=0.4996
  pct_up_pred=0.764  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=30  |  Config=MEDIUM  hidden=64  layers=2

Building datasets (window=30) …
  [train]  55,316 samples |  49 symbols | window=30
  [val  ]   6,027 samples |  49 symbols | window=30
  [test ]   9,095 samples |  50 symbols | window=30

 GRU Quantile  |  window=30  |  config=medium
GRUQuantile(
  (gru): GRU(34, 64, num_layers=2, batch_first=True, dropout=0.2)
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.2, inplace=False)
  (head): Linear(in_features=64, out_features=7, bias=True)
)

 Total params    : 44,743
 Trainable params: 44,743


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

  Early stop at epoch 46

  Val  MAPE=8.4511%  F1_macro=0.4538  Recall_macro=0.4983  DA=0.5022
  Test MAPE=16.7856%  F1_macro=0.4625  Recall_macro=0.5030  DA=0.4942
  pct_up_pred=0.760  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)

  Window=30  |  Config=LARGE  hidden=128  layers=2

Building datasets (window=30) …
  [train]  55,316 samples |  49 symbols | window=30
  [val  ]   6,027 samples |  49 symbols | window=30
  [test ]   9,095 samples |  50 symbols | window=30

 GRU Quantile  |  window=30  |  config=large
GRUQuantile(
  (gru): GRU(34, 128, num_layers=2, batch_first=True, dropout=0.3)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.3, inplace=False)
  (head): Linear(in_features=128, out_features=7, bias=True)
)

 Total params    : 163,207
 Trainable params: 163,207


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]


  Val  MAPE=8.4440%  F1_macro=0.4764  Recall_macro=0.5141  DA=0.5178
  Test MAPE=16.7937%  F1_macro=0.4705  Recall_macro=0.4902  DA=0.4842
  pct_up_pred=0.678  pct_up_true=0.483  (ratio should be ~1.0 for non-degenerate model)


In [11]:
# ─── Best Model Selection ─────────────────────────────────────────────────

# Best per window — lowest val quantile loss (pinball loss on held-out val set)
best_per_window: Dict[int, Dict] = {}
for _w in WINDOWS:
    _wkeys = [k for k in all_results if k[0] == _w]
    if not _wkeys:
        continue
    _bk = min(_wkeys, key=lambda k: all_results[k]['val_ev']['loss'])
    best_per_window[_w] = all_results[_bk]

# Best overall — lowest val quantile loss (no test data used for selection)
best_key = min(
    all_results,
    key=lambda k: all_results[k]['val_ev']['loss']
)
BEST = all_results[best_key]

# Summary table
print(f"\n{'─'*58}")
print('  Best per window (by Val Quantile Loss):')
print(f"  {'Window':>7}  {'Config':>8}  {'Val Loss':>10}  {'Test MAPE':>10}")
print(f"  {'─'*45}")
for _w in WINDOWS:
    if _w not in best_per_window:
        continue
    _r  = best_per_window[_w]
    _vl = _r['val_ev']['loss']
    _tm = _r['test_metrics'].get('MAPE_price', 0)
    _star = ' ⭐' if _r is BEST else ''
    print(f"  {_w:>7d}  {_r['config']:>8s}  {_vl:>9.6f}   {_tm:>9.4f}%{_star}")

bm = BEST['test_metrics']
print(f"\n  BEST OVERALL: window={BEST['window']}, config={BEST['config'].upper()}")
print(f"    MAPE_price    : {bm['MAPE_price']:.4f}%")
print(f"    MSE_pct       : {bm['MSE_pct']:.6f}   (pct-point²)")
print(f"    RMSE_pct      : {bm['RMSE_pct']:.6f}   (pct-points)")
print(f"    MAE_pct       : {bm['MAE_pct']:.6f}   (pct-points)")
print(f"    F1_macro      : {bm['F1_macro']:.4f}")
print(f"    Accuracy      : {bm['Accuracy']:.4f}")
print(f"    Precision_mac : {bm['Precision_macro']:.4f}")
print(f"    Recall_macro  : {bm['Recall_macro']:.4f}")
print(f"    Dir_Acc       : {bm['Directional_Acc']:.4f}")
print(f"    pct_up_pred   : {bm['pct_up_pred']:.4f}  (should be ≈ pct_up_true)")
print(f"    pct_up_true   : {bm['pct_up_true']:.4f}")
print(f"    Weights       : {BEST['save_path']}")
print(f"{'─'*58}")

# Save best model info
_info = {
    'window': BEST['window'], 'config': BEST['config'],
    'save_path': BEST['save_path'],
    'test_metrics': BEST['test_metrics'],
    'val_metrics':  BEST['val_metrics'],
    'quantiles': QUANTILES, 'feature_cols': FEATURE_COLS,
    'timestamp': datetime.now().isoformat(),
}
(OUTPUT_DIR / 'results' / 'best_model_info.json').write_text(
    json.dumps(_info, indent=2, default=str)
)
print(f"Saved: {OUTPUT_DIR / 'results' / 'best_model_info.json'}")



──────────────────────────────────────────────────────────
  Best per window (by Val Quantile Loss):
   Window    Config    Val Loss   Test MAPE
  ─────────────────────────────────────────────
        7     small   0.005102     17.2552%
       10     small   0.005098     17.2338%
       15    medium   0.005109     16.8846%
       30     small   0.005090     16.7944% ⭐

  BEST OVERALL: window=30, config=SMALL
    MAPE_price    : 16.7944%
    MSE_pct       : 2.268905   (pct-point²)
    RMSE_pct      : 1.506288   (pct-points)
    MAE_pct       : 1.065417   (pct-points)
    F1_macro      : 0.4671
    Accuracy      : 0.4996
    Precision_mac : 0.5118
    Recall_macro  : 0.5085
    Dir_Acc       : 0.4996
    pct_up_pred   : 0.7640  (should be ≈ pct_up_true)
    pct_up_true   : 0.4831
    Weights       : /kaggle/working/gru_outputs/checkpoints/w30_small.pt
──────────────────────────────────────────────────────────
Saved: /kaggle/working/gru_outputs/results/best_model_info.json


In [12]:
# ─── Per-Ticker MAPE Table (Best Model, Test Set) ─────────────────────────

def per_ticker_metrics(ev: Dict) -> pd.DataFrame:
    rows = []
    for sym in sorted(np.unique(ev['symbols'])):
        mask = ev['symbols'] == sym
        m    = compute_metrics(
            ev['true'][mask], ev['pred'][mask],
            ev['base_px'][mask], ev['true_px'][mask]
        )
        m['Symbol'] = sym.replace('.NS', '')
        rows.append(m)
    df = pd.DataFrame(rows).set_index('Symbol').sort_values('MAPE_price')
    return df


print(f"\n{'='*60}")
print(f"  Per-Ticker Test Metrics | window={BEST['window']} | config={BEST['config']}")
print(f"{'='*60}")

test_ticker_df = per_ticker_metrics(BEST['test_ev'])
SHOW_COLS = [
    'MAPE_price', 'MSE_pct', 'RMSE_pct', 'MAE_pct',
    'F1_macro', 'Accuracy', 'Precision_macro', 'Recall_macro',
    'Directional_Acc', 'pct_up_pred', 'pct_up_true', 'n_samples'
]

styled = (
    test_ticker_df[SHOW_COLS].style
    .background_gradient(subset=['MAPE_price'],                        cmap='RdYlGn_r')
    .background_gradient(subset=['F1_macro','Accuracy','Directional_Acc'], cmap='RdYlGn')
    .format({c: '{:.4f}' for c in SHOW_COLS if c != 'n_samples'})
    .format({'n_samples': '{:.0f}'})
    .set_caption(
        f"Per-Ticker Test Metrics  |  Best: window={BEST['window']}, cfg={BEST['config']}"
    )
)
display(styled)

test_ticker_df.to_csv(OUTPUT_DIR / 'results' / 'per_ticker_test_metrics.csv')
print(f"\nSaved: {OUTPUT_DIR / 'results' / 'per_ticker_test_metrics.csv'}")


  Per-Ticker Test Metrics | window=30 | config=small


,MAPE_price,MSE_pct,RMSE_pct,MAE_pct,F1_macro,Accuracy,Precision_macro,Recall_macro,Directional_Acc,pct_up_pred,pct_up_true,n_samples
Symbol,,,,,,,,,,,,
BHARTIARTL,1.491317,1.383924,1.176403,0.886488,0.490913,0.508108,0.518881,0.515801,0.508108,0.702703,0.481081,185
NTPC,1.696886,1.320347,1.149063,0.844001,0.490064,0.535135,0.540517,0.527485,0.535135,0.783784,0.513514,185
BEL,1.703508,2.688667,1.639715,1.241620,0.464169,0.497297,0.543919,0.528408,0.497297,0.800000,0.448649,185
SBIN,1.800968,1.843799,1.357865,0.909938,0.491353,0.497297,0.491667,0.491765,0.497297,0.567568,0.540541,185
COALINDIA,1.831860,1.891670,1.375380,0.936675,0.524603,0.562162,0.563184,0.547384,0.562162,0.751351,0.529730,185
SUNPHARMA,1.877832,1.320611,1.149179,0.831144,0.434861,0.540541,0.475327,0.487819,0.540541,0.859459,0.572973,185
ICICIBANK,1.898563,1.214281,1.101944,0.831851,0.432567,0.454054,0.501016,0.500722,0.454054,0.778378,0.416216,185
M&M,1.945990,2.785593,1.669010,1.207002,0.534508,0.545946,0.560000,0.552669,0.545946,0.675676,0.481081,185
POWERGRID,2.024961,1.546783,1.243697,0.885210,0.462209,0.545946,0.585053,0.535673,0.545946,0.881081,0.513514,185



Saved: /kaggle/working/gru_outputs/results/per_ticker_test_metrics.csv


In [13]:
# ─── Actual vs Predicted Plots — TEST only — Reconstructed INR Prices ─────
# 1. Load raw Adj Close prices from nifty50_ticker.csv
# 2. For each test sample: pred_price = raw_price[t-1] * (1 + pred_return)
# 3. Plot actual_price vs predicted_price
# 4. Save INDIVIDUAL plots for every company in a dedicated folder

import pandas as pd

# ── Load raw prices ────────────────────────────────────────────────────────
_ticker_df = pd.read_csv(TICKER_PATH)
_ticker_df['Date'] = pd.to_datetime(_ticker_df['Date'], utc=True).dt.tz_localize(None)
_ticker_df = _ticker_df.sort_values(['Symbol', 'Date'])

# Build a dict: (symbol, date_str) → raw Adj Close
_raw_px = {}
for _, row in _ticker_df.iterrows():
    key = (row['Symbol'], row['Date'].strftime('%Y-%m-%d'))
    _raw_px[key] = float(row['Adj Close'])


def _get_raw_price(sym, dt):
    """Look up raw INR price from ticker CSV."""
    d = pd.Timestamp(dt)
    key = (sym, d.strftime('%Y-%m-%d'))
    return _raw_px.get(key, None)


def save_individual_predictions(ev, window, cfg, folder_name='test_predictions'):
    """
    Saves individual Actual vs Predicted INR Price plots for each company.
    """
    unique_syms = sorted(np.unique(ev['symbols']))
    n_syms = len(unique_syms)
    
    out_dir = OUTPUT_DIR / 'plots' / folder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {n_syms} individual company plots in {out_dir}...")
    
    for sym in unique_syms:
        mask = ev['symbols'] == sym

        dt   = ev['dates'][mask]
        pred = ev['pred'][mask]     # predicted pct_change (fraction)
        true = ev['true'][mask]     # actual pct_change (fraction)

        # Sort by date
        order = np.argsort([pd.Timestamp(d) for d in dt])
        dt   = dt[order]
        pred = pred[order]
        true = true[order]

        # ── Reconstruct INR prices ──────────────────────────────────────
        actual_prices = []
        pred_prices   = []
        valid_dates   = []

        for j in range(len(dt)):
            d = pd.Timestamp(dt[j])
            raw_px_today = _get_raw_price(sym, d)
            if raw_px_today is None:
                continue

            actual_prices.append(raw_px_today)

            actual_ret = true[j]
            denom = 1.0 + actual_ret
            if abs(denom) < 1e-8:
                denom = 1e-8
            prev_raw = raw_px_today / denom
            pred_px  = prev_raw * (1.0 + pred[j])
            
            pred_prices.append(pred_px)
            valid_dates.append(d)

        if len(actual_prices) < 2:
            continue

        actual_prices = np.array(actual_prices)
        pred_prices   = np.array(pred_prices)
        xs = np.arange(len(actual_prices))

        # ── Plot Setup ──
        fig, ax = plt.subplots(figsize=(7, 4.5))

        # Solid dark line = actual, solid blue line = predicted
        ax.plot(xs, actual_prices, lw=1.2, color='#1a1a2e', label='Actual Price',    alpha=0.95)
        ax.plot(xs, pred_prices,   lw=1.2, color='#4361ee', label='Predicted Price', alpha=0.85)

        # X-axis dense date labels (about 6 ticks)
        n_ticks  = min(6, len(xs))
        tick_pos = np.linspace(0, len(xs) - 1, n_ticks, dtype=int)
        tick_lbl = [pd.Timestamp(valid_dates[p]).strftime("%d %b '%y") for p in tick_pos]
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_lbl, rotation=25, ha='right', fontsize=8)

        short = sym.replace('.NS', '')
        ax.set_title(f"{short}\nActual vs Predicted Price (Test Window)", fontsize=11, fontweight='bold', pad=10)
        ax.set_ylabel('Price (INR)', fontsize=9, fontweight='bold')
        ax.tick_params(axis='y', labelsize=8)
        
        ax.legend(fontsize=9, loc='best', framealpha=0.9, edgecolor='#cccccc')
        ax.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout(pad=1.0)
        
        # Save individual file
        out_file = out_dir / f"{short}.png"
        plt.savefig(out_file, bbox_inches='tight', dpi=150)
        plt.close(fig)

    print(f"✅ Successfully saved {n_syms} plot images to {out_dir}")


# ── Plot ONLY the best model's TEST predictions (Individual Images) ──────
save_individual_predictions(
    BEST['test_ev'], BEST['window'], BEST['config'],
    folder_name=f"best_model_test_w{BEST['window']}_{BEST['config']}"
)


Saved: /kaggle/working/gru_outputs/plots/best_model_TRAIN_predictions.png
Saved: /kaggle/working/gru_outputs/plots/best_model_VAL_predictions.png
Saved: /kaggle/working/gru_outputs/plots/best_model_TEST_predictions.png


In [14]:
# ─── Per-Window Best-Model Metrics Comparison ─────────────────────────────

def plot_window_metrics(bpw, save_path):
    _metrics = [
        ('MAPE on Price (%)',     'MAPE_price'),
        ('MAE (pct-points)',      'MAE_pct'),
        ('RMSE (pct-points)',     'RMSE_pct'),
        ('F1 Score (macro)',      'F1_macro'),
        ('Accuracy',             'Accuracy'),
        ('Directional Acc',      'Directional_Acc'),
    ]
    ws   = sorted(bpw.keys())
    cfgs = [bpw[w]['config'] for w in ws]
    xlbs = [f'W{w}\n({c})' for w, c in zip(ws, cfgs)]
    clrs = [PALETTE[i] for i in range(len(ws))]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, (label, key) in zip(axes.ravel(), _metrics):
        vals = [bpw[w]['test_metrics'].get(key, 0) for w in ws]
        bars = ax.bar(xlbs, vals, color=clrs, edgecolor='white', lw=0.8, width=0.65)
        for b, v in zip(bars, vals):
            ax.text(
                b.get_x() + b.get_width() / 2, b.get_height(),
                f'{v:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold'
            )
        ax.set_title(label, fontsize=9, fontweight='bold')
        ax.set_ylabel(key.replace('_', ' '), fontsize=7)
        ax.tick_params(axis='x', labelsize=7)
        ax.set_ylim(0, max(vals) * 1.25 + EPS)

    fig.suptitle('Per-Window Best GRU Model — Test Set Metrics',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=110)
    plt.show(); plt.close()
    print(f'Saved: {save_path}')


plot_window_metrics(
    best_per_window,
    OUTPUT_DIR / 'plots' / 'per_window_best_metrics.png'
)

Saved: /kaggle/working/gru_outputs/plots/per_window_best_metrics.png


In [15]:
# ─── Train / Val Loss Curves ──────────────────────────────────────────────

def plot_loss_curves(histories, bpw, save_path):
    ws       = WINDOWS
    cfg_list = list(MODEL_CONFIGS.keys())
    nw, nc   = len(ws), len(cfg_list)

    fig, axes = plt.subplots(nw, nc, figsize=(5.5 * nc, 3.2 * nw), squeeze=False)

    for row, w in enumerate(ws):
        best_cfg = bpw[w]['config'] if w in bpw else None
        for col, cfg in enumerate(cfg_list):
            ax  = axes[row][col]
            key = (w, cfg)
            if key not in histories:
                ax.set_visible(False); continue
            h      = histories[key]
            ep     = range(1, len(h['train_loss']) + 1)
            ax.plot(ep, h['train_loss'], lw=1.3, color=PALETTE[0], label='Train')
            ax.plot(ep, h['val_loss'],   lw=1.3, color=PALETTE[1], label='Val', ls='--')
            is_best = cfg == best_cfg
            title   = f'W={w} | {cfg.upper()}' + (' ⭐' if is_best else '')
            ax.set_title(title, fontsize=8, fontweight='bold' if is_best else 'normal')
            ax.set_xlabel('Epoch', fontsize=7)
            ax.set_ylabel('Quantile Loss', fontsize=7)
            ax.legend(fontsize=7)

    fig.suptitle('Train / Val Quantile Loss  —  All (Window × Config) — GRU',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=110)
    plt.show(); plt.close()
    print(f'Saved: {save_path}')


plot_loss_curves(
    all_histories, best_per_window,
    OUTPUT_DIR / 'plots' / 'train_val_loss_curves.png'
)

Saved: /kaggle/working/gru_outputs/plots/train_val_loss_curves.png


In [16]:
# ─── Final Results Table ──────────────────────────────────────────────────

rows = []
for (w, cfg) in sorted(all_results):
    r  = all_results[(w, cfg)]
    tm = r['test_metrics']
    vm = r['val_metrics']
    rows.append({
        'Window':          w,
        'Config':          cfg,
        'Val MAPE (%)':    round(vm.get('MAPE_price',      0), 4),
        'Test MAPE (%)':   round(tm.get('MAPE_price',      0), 4),
        'MAE (pct-pts)':   round(tm.get('MAE_pct',         0), 4),
        'RMSE (pct-pts)':  round(tm.get('RMSE_pct',        0), 4),
        'F1_macro':        round(tm.get('F1_macro',         0), 4),
        'Accuracy':        round(tm.get('Accuracy',         0), 4),
        'Prec_macro':      round(tm.get('Precision_macro',  0), 4),
        'Recall_macro':    round(tm.get('Recall_macro',     0), 4),
        'DirectionalAcc':  round(tm.get('Directional_Acc',  0), 4),
        'pct_up_pred':     round(tm.get('pct_up_pred',      0), 3),
        'pct_up_true':     round(tm.get('pct_up_true',      0), 3),
        'Best?':           '⭐' if (w, cfg) == best_key else '',
    })

df_all = pd.DataFrame(rows)

display(
    df_all.style
    .background_gradient(subset=['Test MAPE (%)'],             cmap='RdYlGn_r')
    .background_gradient(subset=['F1_macro', 'DirectionalAcc'], cmap='RdYlGn')
    .set_caption('All Hyperparameter Configurations — Test Metrics (GRU)')
)

df_all.to_csv(OUTPUT_DIR / 'results' / 'all_configs_summary.csv', index=False)

print(f"\n{'='*60}")
print('  OUTPUT FILES')
print(f"{'='*60}")
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        size = p.stat().st_size
        print(f'  {str(p.relative_to(OUTPUT_DIR)):<55} {size/1024:.1f} KB')

,Window,Config,Val MAPE (%),Test MAPE (%),MAE (pct-pts),RMSE (pct-pts),F1_macro,Accuracy,Prec_macro,Recall_macro,DirectionalAcc,pct_up_pred,pct_up_true,Best?
0,7,large,8.454900,17.244500,1.061200,1.504400,0.449700,0.489800,0.499300,0.499600,0.489800,0.787000,0.483000,
1,7,medium,8.445500,17.206300,1.057700,1.495900,0.478900,0.493200,0.499300,0.499400,0.493200,0.682000,0.483000,
2,7,small,8.481300,17.255200,1.065200,1.506900,0.480100,0.499800,0.508500,0.507000,0.499800,0.711000,0.483000,
3,10,large,8.448400,17.193500,1.057700,1.497500,0.496500,0.499200,0.497200,0.497300,0.499200,0.444000,0.483000,
4,10,medium,8.451100,17.208500,1.065400,1.505600,0.485400,0.487400,0.489900,0.490100,0.487400,0.580000,0.483000,
5,10,small,8.470500,17.233800,1.063800,1.507600,0.484700,0.502800,0.511600,0.509700,0.502800,0.704000,0.483000,
6,15,large,8.442500,16.895800,1.067200,1.510500,0.399700,0.488100,0.504500,0.501600,0.488100,0.901000,0.483000,
7,15,medium,8.451200,16.884600,1.059100,1.502100,0.490500,0.492000,0.494200,0.494300,0.492000,0.570000,0.483000,
8,15,small,8.442800,16.870500,1.058800,1.498800,0.486600,0.499000,0.505400,0.504800,0.499000,0.672000,0.483000,
9,30,large,8.444000,16.793700,1.059200,1.501600,0.470500,0.484200,0.488800,0.490200,0.484200,0.678000,0.483000,



  OUTPUT FILES
  checkpoints/w10_large.pt                                640.9 KB
  checkpoints/w10_medium.pt                               178.1 KB
  checkpoints/w10_small.pt                                29.6 KB
  checkpoints/w15_large.pt                                640.9 KB
  checkpoints/w15_medium.pt                               178.1 KB
  checkpoints/w15_small.pt                                29.6 KB
  checkpoints/w30_large.pt                                640.9 KB
  checkpoints/w30_medium.pt                               178.1 KB
  checkpoints/w30_small.pt                                29.6 KB
  checkpoints/w7_large.pt                                 640.8 KB
  checkpoints/w7_medium.pt                                178.0 KB
  checkpoints/w7_small.pt                                 29.6 KB
  plots/best_model_TEST_predictions.png                   1746.5 KB
  plots/best_model_TRAIN_predictions.png                  1644.7 KB
  plots/best_model_VAL_predictions.png          